# No Peeking: reproduce on Kaggle (free T4 GPU)

Settings: *Accelerator = GPU T4*, *Internet = on*. The v1 dataset is downloaded from the GitHub release, so no SEC access is needed.
Each step caches its outputs: re-run the notebook after a session timeout.

In [ ]:
!git clone -q https://github.com/KvnFltr/no-peeking.git
%cd no-peeking
!pip -q install -e ".[train]" bitsandbytes
!mkdir -p data/processed/v1 && cd data/processed/v1 && for f in nopeek8k.parquet stats.json filter_log.json; do wget -q https://github.com/KvnFltr/no-peeking/releases/download/data-v1/$f; done

## 1. Look-ahead probe (RQ1)
Models and dtypes come from `configs/models.yaml` (7B+ in 4-bit to fit a 16 GB T4).

In [ ]:
import yaml
models = yaml.safe_load(open('configs/models.yaml'))['models']
for m, meta in models.items():
    dtype = '4bit' if meta['dtype'] == '4bit' else 'float16'
    !nopeeking -v probe {m} --config configs/v1.yaml --limit 2000 --max-input-tokens 1024 --batch-size 8 --dtype {dtype}

## 2. Hindsight rationales (RQ2), teacher Qwen2.5-7B-Instruct in 4-bit

In [ ]:
!nopeeking -v rationales --config configs/train/rationales.yaml
!cat data/processed/v1/rationales/*_summary.json

## 3. Post-training (LoRA) and evaluation on validation + post-cutoff test

In [ ]:
for run in ['sft_direct', 'sft_control', 'sft_hindsight', 'dpo', 'grpo_outcome', 'grpo_grounded', 'grpo_full']:
    !nopeeking -v train {run.split('_')[0]} --config configs/train/{run}.yaml
    !nopeeking eval-adapter --config configs/train/{run}.yaml

## 4. Report (leaderboard, tables, figure, LaTeX macros)

In [ ]:
!nopeeking report --config configs/v1.yaml
from IPython.display import Image, Markdown
display(Image('results/v1/figures/hero.png'))
display(Markdown(open('results/v1/summary.md').read()))